# lab3 homework

softmax classifier trained with mini batch gradient descent.

In [ ]:
train_path = "/kaggle/input/competitions/nn-2026-fii-homework-1/train.npz"
test_path = "/kaggle/input/competitions/nn-2026-fii-homework-1/test.npz"

## helpers

In [ ]:
import numpy as np

def load_dataset(path):
  data = np.load(path)
  x = data["data"]
  y = data["labels"] if "labels" in data.files else None
  return x, y


def split_dataset(x, y, val_fraction=0.1):
  split = int(len(x) * (1 - val_fraction))
  x_train, x_val = x[:split], x[split:]
  y_train, y_val = y[:split], y[split:]
  return x_train, y_train, x_val, y_val

def load_parameters(path):
  params = np.load(path)
  w = params["w"]
  b = params["b"]

  return w, b

def prepare_trainset(x_train):
  n_features = len(x_train[0])
  n_classes = 10

  return n_features, n_classes

def normalize_x(x):
  x_float = x.astype(np.float32)
  return x_float / 255

def normalize_y(y, nr_classes):
  ln = len(y)
  norm_y = np.zeros((ln, nr_classes))
  rows = np.arange(ln)
  norm_y[rows, y] = 1

  return norm_y

def mse(y, z):
  return np.mean((y - z) ** 2)

def cross_entropy(y, z):
  return -np.mean(np.sum(y * np.log(z + 1e-12), axis=1))

def update_params(current_w, current_b, lr, err, x):
  updated_w = current_w + lr * x.transpose() @ err / len(x)
  updated_b = current_b + lr * err.sum(axis=0) / len(x)
  return updated_w, updated_b

def softmax(z):
  row_max = np.max(z, axis=1, keepdims=1)
  new_z = np.exp(z - row_max) / np.sum(np.exp(z - row_max), axis = 1, keepdims=True)

  return new_z

In [ ]:
import numpy as np

def make_params(nr_features, nr_classes):
    w = np.zeros((nr_features, nr_classes))
    b = np.zeros(nr_classes)
    lr = 0.07

    def get_params():
        return w, b, lr

    def set_params(new_w, new_b):
        nonlocal w, b
        w, b = new_w, new_b

    return get_params, set_params

In [ ]:
def perceptron(x, w, b):
  return x @ w + b

## train

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def train(x, y, read_params, save_params, epochs=200, batch_size=64):
  history = []
  times = []
  start_time = time.perf_counter()

  for epoch in range(epochs):
    for start in range(0, len(x), batch_size):
      batch_x = x[start:start+batch_size]
      batch_y = y[start:start+batch_size]

      w, b, lr = read_params()
      z = perceptron(batch_x, w, b)
      t = softmax(z)
      e = batch_y - t
      new_w, new_b = update_params(w, b, lr, e, batch_x)
      save_params(new_w, new_b)

    w, b, lr = read_params()
    z = perceptron(x, w, b)
    t = softmax(z)
    epoch_loss = cross_entropy(y, t)
    history.append(epoch_loss)
    times.append(time.perf_counter() - start_time)
    print(f"epoch {epoch + 1}: loss = {epoch_loss:.5f}, time = {times[-1]:.1f}s")
  return history, times

def plot_loss_vs_time(loss_history, times):
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
  
  epochs = range(1, len(loss_history) + 1)

  ax1.plot(epochs, loss_history, marker="o")
  ax1.set_xlabel("epoch")
  ax1.set_ylabel("Cross-entropy on training data")
  ax1.set_title("Error per epoch")
  ax1.grid(alpha=0.3)

  ax2.plot(epochs, times, marker="o")
  ax2.set_xlabel("epoch")
  ax2.set_ylabel("time since start (seconds)")
  ax2.set_title("Training time")
  ax2.grid(alpha=0.3)

  fig.tight_layout()
  fig.savefig("training_curve.png", dpi=150)

def main():
  x, y = load_dataset(train_path)
  x_train, y_train, _, _ = split_dataset(x, y)
  nr_features, nr_classes = prepare_trainset(x_train)
  get_params, set_params = make_params(nr_features, nr_classes)
  normalized_x = normalize_x(x_train)
  normalized_y = normalize_y(y_train, nr_classes)
  history, times = train(normalized_x, normalized_y, get_params, set_params)
  w, b, _ = get_params()
  np.savez("model.npz", w=w, b=b)
  plot_loss_vs_time(history, times)

main()

## eval

In [ ]:
import numpy as np

def evaluate():
  x, y = load_dataset(train_path)
  _, _, x_val, y_val = split_dataset(x, y)

  w, b = load_parameters("model.npz")
  normalized_x = normalize_x(x_val)
  z = perceptron(normalized_x, w, b)

  predictions = np.argmax(z, axis=1)
  acc = np.mean(predictions == y_val)

  print(f"Accuracy: {acc * 100:.2f}%")

evaluate()

## test

In [ ]:
import numpy as np

def save_submission(predictions, path="submission.csv"):
  ids = np.arange(len(predictions))
  rows = np.column_stack((ids, predictions))
  np.savetxt(path, rows, fmt="%d", delimiter=",", header="ID,target", comments="")

def predict():
  x, _ = load_dataset(test_path)

  w, b = load_parameters("model.npz")
  normalized_x = normalize_x(x)

  z = perceptron(normalized_x, w, b)

  predictions = np.argmax(z, axis=1)

  print(f"Predictions: {predictions}") 
  save_submission(predictions)

predict()